# Execução do RAG no Google Colab

Este notebook foi preparado para rodar o sistema RAG utilizando a estrutura de pastas do projeto. Ele assume que você já tem a Base de Conhecimento (KB) montada e quer apenas realizar consultas e experimentos.

## 1. Conexão com o Ambiente e Dependências

In [1]:
# faz o download do livro pelo notebook python do Colab
!wget https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf

--2026-05-10 17:02:43--  https://domainpublic.wordpress.com/wp-content/uploads/2022/10/jk_couto_2ed.pdf
Resolving domainpublic.wordpress.com (domainpublic.wordpress.com)... 192.0.78.12, 192.0.78.13
Connecting to domainpublic.wordpress.com (domainpublic.wordpress.com)|192.0.78.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16321321 (16M) [application/pdf]
Saving to: ‘jk_couto_2ed.pdf’

jk_couto_2ed.pdf    100%[===================>]  15.56M  --.-KB/s    in 0.08s   

2026-05-10 17:02:43 (199 MB/s) - ‘jk_couto_2ed.pdf’ saved [16321321/16321321]



In [2]:
!pip install langchain langchain-huggingface langchain-community langchain-experimental langchain-core langchain-classic pydantic langchain-chroma chromadb sentence-transformers pypdf pymupdf python-dotenv rank-bm25 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2

In [3]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HuggingFaceKey'))

import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HuggingFaceKey')

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
import re



clean_text = 1

def limpar_texto(texto):
    texto = re.sub(r'­\n', '', texto)      # hifenização invisível
    texto = re.sub(r'-\n', '', texto)       # hifenização normal
    texto = re.sub(r'\d+\s*•\s*JK\s*\n', '', texto)   # "430 • JK"
    texto = re.sub(r'JK\s*•\s*\d+\s*\n', '', texto)   # "JK • 183"
    texto = re.sub(r'(?<!\n)\n(?!\n)', ' ', texto)     # quebras no meio de frase
    texto = re.sub(r' +', ' ', texto)
    return texto.strip()

def initialize_kb(embeddings_model):
    vector_store = Chroma(
        collection_name="embeddings_jk",
        embedding_function=embeddings_model,
        persist_directory="./chroma_db"
    )

    if vector_store._collection.count() == 0:
        # Só carrega PDF e processa se realmente precisar

        arquivo = 'jk_couto_2ed.pdf'
        loader = PyMuPDFLoader(f"{arquivo}")
        all_pages = loader.load()

        if clean_text == 0:
            for doc in all_pages:
                doc.page_content = limpar_texto(doc.page_content)

        text_splitter = SemanticChunker(
            embeddings=embeddings_model,
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=85
        )

        chunks = text_splitter.split_documents(all_pages)
        print(f"Documento dividido em {len(chunks)} partes.")
        vector_store.add_documents(documents=chunks)
    else:
        # Execuções seguintes: recupera do disco para o BM25
        print("KB já existente, carregando do disco.")
        result = vector_store.get()
        chunks = [
            Document(page_content=text, metadata=meta)
            for text, meta in zip(result["documents"], result["metadatas"])
        ]
        print(f"{len(chunks)} chunks recuperados do disco.")

    return vector_store, chunks



In [5]:
content = """
Você é um especialista em análise de documentos. Sua tarefa é responder à pergunta do usuário usando APENAS as informações contidas nos <trechos_fornecidos>.

Regras:
1. Identifique qual alternativa é correta com base nos trechos fornecidos.
2. Justifique a escolha com uma explicação dissertativa e objetiva, Você pode e deve usaSSSr breves citações literais.
3. Se os trechos comprovarem a ideia central de uma alternativa e invalidarem as demais, marque-a como correta mesmo que não cubra todos os detalhes.
4. Nunca utilize conhecimentos externos aos trechos fornecidos.
5. Se nenhuma alternativa puder ser confirmada pelos trechos, responda: "Alternativa Não encontrado no contexto."

### Exemplos

---
Pergunta: Em que ano o governador assinou o decreto e qual era o objetivo principal?
A) O decreto foi assinado em 1960 para modernizar o sistema tributário.
B) O decreto foi assinado em 1962 para construir estradas no interior.
C) O decreto foi assinado em 1962 para reformar o sistema educacional.
D) O decreto foi assinado em 1958 para ampliar portos marítimos.
E) O decreto não foi assinado, pois as negociações fracassaram.

<trechos_fornecidos>
[Trecho 1]: A assembleia legislativa debateu por meses a proposta apresentada pelo executivo estadual. Havia divergências sobre os prazos e os recursos necessários para a implementação.
[Trecho 2]: Após intensas negociações, o governador assinou o decreto em 1962, estabelecendo as diretrizes para a construção de novas estradas no interior do estado.
[Trecho 3]: A medida foi recebida com entusiasmo pelas comunidades rurais, que há anos aguardavam melhorias na infraestrutura de transporte da região.
</trechos_fornecidos>

Alternativa Correta: B - Apesar das divergências iniciais na assembleia, as negociações foram concluídas e o decreto foi assinado em 1962 com foco na expansão viária do interior, atendendo à demanda das comunidades rurais por melhor infraestrutura de transporte.

---
Pergunta: Qual era a principal fonte de financiamento do projeto de modernização portuária?
A) O projeto foi financiado exclusivamente por investidores estrangeiros.
B) O financiamento veio de uma parceria entre o governo federal e os municípios costeiros.
C) O projeto foi custeado por um fundo internacional de desenvolvimento.
D) O financiamento foi garantido por meio de títulos públicos emitidos pelo governo estadual.
E) O projeto contou com recursos provenientes de uma taxa cobrada sobre exportações.

<trechos_fornecidos>
[Trecho 1]: A modernização dos portos foi debatida em diversas conferências regionais ao longo da década, com a participação de representantes do setor privado e de autoridades municipais.
[Trecho 2]: Os debates giravam em torno da necessidade de ampliar a capacidade de escoamento da produção agrícola, considerada insuficiente para atender à demanda crescente dos mercados externos.
[Trecho 3]: Diversas propostas foram apresentadas, mas nenhuma decisão formal sobre o modelo de financiamento havia sido tomada até o encerramento do período analisado.
</trechos_fornecidos>

Alternativa Correta: Não encontrado no contexto - Os trechos descrevem os debates e a motivação para a modernização portuária, mas não fornecem informações sobre o modelo de financiamento adotado, tornando impossível confirmar qualquer uma das alternativas apresentadas.

---
<trechos_fornecidos>
{contexto}
</trechos_fornecidos>

Pergunta e Alternativas:
{pergunta}

Formato de Resposta Obrigatório:
Alternativa Correta: [Letra] - Justificativa dissertativa da escolha
"""

In [6]:
from langchain_core.language_models.chat_models import BaseChatModel
from abc import ABC, abstractmethod


class BaseRAG(ABC):
    """
    Interface base para implementações de sistemas RAG.
    """

    def __init__(self, llm_instance: BaseChatModel, **kwargs):
        """
        Args:
            llm_instance: Instância pré-carregada de um modelo de linguagem BaseChatModel (ex.: ChatOpenAI, ChatHuggingFace, etc.)
            **kwargs: Parâmetros adicionais (temperature, max_new_tokens, etc.).
        """
        assert isinstance(llm_instance, BaseChatModel), "llm_instance deve ser uma instância de BaseChatModel"
        self.llm_instance = llm_instance
        self.params = kwargs

    # função auxiliar
    def _generate_response(self, system_prompt: str, user_prompt: str) -> str:
        """
        Gera uma resposta para a pergunta fornecida usando o modelo de linguagem.

        Args:
            system_prompt (str): O prompt do sistema.
            user_prompt (str): O prompt do usuário.

        Returns:
            str: A resposta gerada pelo modelo de linguagem.
        """
        if system_prompt is None or system_prompt.strip() == "":
            chat_history = [
                ("user", user_prompt)
            ]
        else:
            chat_history = [
                ("system", system_prompt),
                ("user", user_prompt)
            ]
        response = self.llm_instance.invoke(chat_history, **self.params)
        return response.text

    @abstractmethod
    def answer_question(self, question: str) -> str:
        """
        Processa uma pergunta e retorna a resposta gerada pelo RAG.

        Args:
            question (str): A pergunta a ser respondida.

        Returns:
            str: A resposta gerada pelo RAG.
        """
        pass

    @abstractmethod
    def teardown(self) -> None:
        """
        Libera os recursos de hardware utilizados especificamente pelo RAG.

        Deve limpar a base de vetores e referências a documentos da memória,
        garantindo que o hardware esteja limpo para a próxima execução.
        """
        pass

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.retrievers import BaseRetriever
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder

from pydantic import Field, ConfigDict
from langchain_core.documents import Document
from typing import Any, List

import torch

device = ''

if(torch.cuda.is_available()):
    device = "cuda"
else:
    device = "cpu"

class MyRAG(BaseRAG):
    def __init__(self, llm_instance, param1=None, param2=None, **kwargs):
        self.embeddings_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={"device": device}
        )
        self.vector_store, self.chunks = initialize_kb(embeddings_model=self.embeddings_model)
        self.bm25 = BM25Retriever.from_documents(self.chunks, k=5)
        self.cross_encoder = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
        self.prompt = PromptTemplate(
            input_variables=["contexto", "pergunta"],
            template=content
        )
        super().__init__(llm_instance, **kwargs)

    # Retriever com MMR e EnsembleRetriever
    def _build_retriever(self, quantity: int = 5):
        semantic = self.vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": quantity, "fetch_k": 30, "lambda_mult": 0.6}
        )

        ensemble = EnsembleRetriever(
            retrievers=[self.bm25, semantic],
            weights=[0.5, 0.5]
        )

        cross_encoder = self.cross_encoder

        # Cross-encoder -> Principal peça para o re-ranking
        class CrossEncoderRetriever(BaseRetriever):
            vectorstore: Any
            cross_encoder: Any
            k: int = Field(default=10)
            rerank_top_k: int = Field(default=3)

            model_config = ConfigDict(arbitrary_types_allowed=True)

            def _get_relevant_documents(self, query: str, *, run_manager) -> List[Document]:
                initial_docs = ensemble.invoke(query)
                pairs = [[query, doc.page_content] for doc in initial_docs]
                scores = self.cross_encoder.predict(pairs)
                scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
                return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

            async def _aget_relevant_documents(self, query: str, *, run_manager) -> List[Document]:
                raise NotImplementedError("Async não implementado")

        return CrossEncoderRetriever(
            vectorstore=self.vector_store,
            cross_encoder=cross_encoder,
            k=quantity * 2,
            rerank_top_k=3
        )


    # Função específica pra rodar o RAG workflow
    def answer_question(self, question: str, mostrar_chunks: bool = False) -> str:
        retriever = self._build_retriever()
        docs = retriever.invoke(question)

        if mostrar_chunks:
            print("----")
            for i, doc in enumerate(docs):
                print(f" => chunk {i+1}: {doc.page_content[:80]}...")
            print("----")

        contexto = "\n".join(
            f"[Trecho {i+1}]: {doc.page_content}"
            for i, doc in enumerate(docs)
        )
        final_prompt = self.prompt.format(contexto=contexto, pergunta=question)

        response = self._generate_response(system_prompt='', user_prompt=final_prompt)

        # Caso seja interessante mostrar os chunks
        if mostrar_chunks:
            return response, [d.page_content for d in docs]

        return response

    # Limpar a rodagem do RAG
    def teardown(self) -> None:
        self.vector_store._client.close()

        del self.vector_store
        del self.chunks
        del self.bm25
        del self.cross_encoder

        print("Recursos de hardware liberados.")


## 2. Inicialização do Modelo (LLM)

Aqui instanciamos o modelo que será usado para gerar as respostas. No exemplo abaixo, usamos o `Gemma-2b-it` via HuggingFace.

In [7]:
MODEL_ID = "google/gemma-3-4b-it"

local_llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs=dict(
        do_sample=True,
        max_new_tokens=2048,
        return_full_text=False
    )
)
chat_model = ChatHuggingFace(llm=local_llm)
rag = MyRAG(llm_instance=chat_model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

## 3. Realizando Perguntas (Experimentos)

Agora você pode testar o seu RAG com diferentes perguntas.

In [9]:
question = """
        "Qual a data exata do falecimento de JK?\n"
        "A) 24 de agosto de 1954\n"
        "B) 21 de agosto de 1924\n"
        "C) 12 de setembro de 1902\n"
        "D) 8 de junho de 1964"
    """

resposta, chunks = rag.answer_question(question, mostrar_chunks=True) # é opcional essa função de mostrar chunks, só para esclarecer melhor os dados recuperados.

print("\n--- RESPOSTA DO RAG ---\n")
print(resposta)

Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----
 => chunk 1: 448 • JK
Artigo de Adolpho Bloch sobre JK
24 de agosto de 1976
Fonte: Revista Ma...
 => chunk 2: JK • 427
Nota de JK à nação
25 de maio de 1964
Fonte: Folha de S. Paulo, 26 de m...
 => chunk 3: JK • 243
Capítulo 17
A morte na curva da estrada
JK, no diário, sobre a virada d...
----

--- RESPOSTA DO RAG ---

Alternativa Correta: D - O Trecho 3 menciona que Juscelino Kubitschek (JK) sentia-se bem ao ver nascer 1976, e também cita uma entrada no diário dele sobre a virada de 1975 para 1976. Embora o trecho não forneça a data exata de sua morte, ele está relacionado ao período de 1975-1976, o que implica que JK já faleceu antes desse período. A alternativa D, 8 de junho de 1964, é a única que se encaixa nesse contexto temporal, sendo a data de seu falecimento.


In [10]:
chunks

['448 • JK\nArtigo de Adolpho Bloch sobre JK\n24 de agosto de 1976\nFonte: Revista Manchete nº 2.534, Rio de Janeiro, fevereiro de 2006. Nos últimos 13 anos fomos companheiros inseparáveis.',
 'JK • 427\nNota de JK à nação\n25 de maio de 1964\nFonte: Folha de S. Paulo, 26 de maio de 1964, p.',
 'JK • 243\nCapítulo 17\nA morte na curva da estrada\nJK, no diário, sobre a virada de 1975 para 1976:\n“Vimos nascer 1976. Sentia-me bem.']

## 4. Finalização

Sempre libere os recursos ao terminar para não estourar a memória do Colab.

In [12]:
import torch, gc

rag.teardown()
gc.collect()
torch.cuda.empty_cache()

Recursos de hardware liberados.
